# 03. 분류 모델 학습 (Classification Model)

**실행 환경:** Kaggle GPU (T4/P100)  
**목적:** KoBERT 파인튜닝으로 민원 분류 모델 학습

---

## 모델 전략

| Task | 모델 | 출력 |
|------|------|------|
| 도메인 분류 | KoBERT | domain_id |
| 카테고리 분류 | KoBERT | category_id |
| 의도 분류 | KoBERT | intent_id |

In [1]:
# Kaggle 환경 설정
import os
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

# 필수 패키지 설치
!pip install -q transformers datasets accelerate scikit-learn

In [2]:
import torch
import pandas as pd
import numpy as np
import json
from pathlib import Path
from sklearn.metrics import accuracy_score, f1_score, classification_report
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)
from datasets import Dataset, DatasetDict
import warnings
warnings.filterwarnings('ignore')

# GPU 확인
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

2026-01-10 06:27:19.366447: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1768026439.559792      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1768026439.614101      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1768026440.094214      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1768026440.094251      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1768026440.094258      55 computation_placer.cc:177] computation placer alr

Device: cuda
GPU: Tesla P100-PCIE-16GB


In [3]:
# 경로 설정 (Kaggle Dataset으로 업로드 후 수정)
DATA_PATH = Path('/kaggle/input/kobert')
OUTPUT_PATH = Path('/kaggle/working/models')
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

In [4]:
# 데이터 로드 (Parquet)
train_df = pd.read_parquet(DATA_PATH / 'train_classification.parquet')
val_df = pd.read_parquet(DATA_PATH / 'val_classification.parquet')
test_df = pd.read_parquet(DATA_PATH / 'test_classification.parquet')

# Stratified 샘플링 (클래스 비율 유지)
from sklearn.model_selection import train_test_split

SAMPLE_SIZE = 200000

if len(train_df) > SAMPLE_SIZE:
    train_df, _ = train_test_split(
        train_df, 
        train_size=SAMPLE_SIZE, 
        stratify=train_df['domain_id'],
        random_state=42
    )
    print(f"⚡ Stratified 샘플링: {SAMPLE_SIZE:,}건")

# 레이블 매핑 로드
with open(DATA_PATH / 'label_mapping.json', 'r', encoding='utf-8') as f:
    label_mapping = json.load(f)

import joblib
label_encoders = joblib.load(DATA_PATH / 'label_encoders.joblib')

print(f"Train: {len(train_df):,}, Val: {len(val_df):,}, Test: {len(test_df):,}")

# 클래스 분포 확인
print(f"\n도메인 분포:")
print(train_df['domain_id'].value_counts().sort_index())

⚡ Stratified 샘플링: 200,000건
Train: 200,000, Val: 35,223, Test: 35,224

도메인 분포:
domain_id
0     49710
1      1275
2     45795
3     23820
4       163
5      2554
6      3675
7      4331
8      1952
9      4365
10     4428
11    54144
12     2231
13     1557
Name: count, dtype: int64


In [5]:
# 학습 태스크 & 모델 설정
TASK = 'domain'  # 'domain', 'category', 'intent'
MODEL_NAME = 'klue/bert-base'  # 또는 'monologg/kobert'
MAX_LENGTH = 128

LABEL_COL = f'{TASK}_id'
NUM_LABELS = len(label_mapping[TASK])
print(f"Task: {TASK}, Labels: {NUM_LABELS}")

Task: domain, Labels: 14


In [6]:
from transformers import DataCollatorWithPadding

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def prepare_dataset(df):
    return Dataset.from_pandas(df[['text', LABEL_COL]].rename(columns={LABEL_COL: 'labels'}))

def tokenize_fn(examples):
    return tokenizer(examples['text'], truncation=True, max_length=MAX_LENGTH)
    # ⭐ padding 제거!

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)  # ⭐ 추가!

dataset = DatasetDict({
    'train': prepare_dataset(train_df),
    'validation': prepare_dataset(val_df),
    'test': prepare_dataset(test_df)
})

tokenized = dataset.map(tokenize_fn, batched=True, remove_columns=['text'])
print(tokenized)

tokenizer_config.json:   0%|          | 0.00/289 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/425 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

Map:   0%|          | 0/200000 [00:00<?, ? examples/s]

Map:   0%|          | 0/35223 [00:00<?, ? examples/s]

Map:   0%|          | 0/35224 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['labels', '__index_level_0__', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 200000
    })
    validation: Dataset({
        features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 35223
    })
    test: Dataset({
        features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 35224
    })
})


In [11]:
from torch import nn
from sklearn.utils.class_weight import compute_class_weight

# ⭐ sklearn 방식 클래스 가중치 (더 안정적)
weights = compute_class_weight(
    class_weight='balanced', 
    classes=np.unique(train_df[LABEL_COL]), 
    y=train_df[LABEL_COL]
)
class_weights = torch.tensor(weights, dtype=torch.float32).to(device)

print("클래스 가중치:")
for i, w in enumerate(class_weights):
    label_name = label_mapping[TASK].get(str(i), f'class_{i}')
    print(f"  {i} ({label_name}): {w:.3f}")

# 커스텀 Trainer (가중치 적용)
class WeightedTrainer(Trainer):
    def __init__(self, class_weights, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights
    
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fn = nn.CrossEntropyLoss(weight=self.class_weights)
        loss = loss_fn(logits, labels)
        return (loss, outputs) if return_outputs else loss

# 모델 로드
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=NUM_LABELS, ignore_mismatched_sizes=True
).to(device)

# 메트릭 함수
def compute_metrics(eval_pred):
    preds = np.argmax(eval_pred.predictions, axis=1)
    labels = eval_pred.label_ids
    return {
        'accuracy': accuracy_score(labels, preds),
        'f1_macro': f1_score(labels, preds, average='macro'),
        'f1_weighted': f1_score(labels, preds, average='weighted')
    }

클래스 가중치:
  0 (K쇼핑): 0.287
  1 (관광여가오락): 11.204
  2 (금융/보험): 0.312
  3 (다산콜센터): 0.600
  4 (부동산): 87.642
  5 (부동산업): 5.593
  6 (생활서비스): 3.887
  7 (소매): 3.298
  8 (숙박): 7.319
  9 (음식점): 3.273
  10 (의복의류점): 3.226
  11 (질병관리본부): 0.264
  12 (카페): 6.403
  13 (학원): 9.175


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at klue/bert-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [12]:
args = TrainingArguments(
    output_dir=str(OUTPUT_PATH / f'{TASK}_classifier'),
    
    # 학습 설정
    num_train_epochs=3,
    per_device_train_batch_size=128,
    per_device_eval_batch_size=256,
    
    # ⭐ Kaggle 안정성: num_workers=0
    dataloader_num_workers=0,
    dataloader_pin_memory=True,
    
    # 최적화
    fp16=True,
    optim='adamw_torch_fused',
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    lr_scheduler_type='cosine',
    
    # 평가/저장
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1_macro',
    greater_is_better=True,
    
    # 로깅
    logging_steps=100,
    report_to='none',
    
    seed=42
)

# WeightedTrainer 사용
trainer = WeightedTrainer(
    class_weights=class_weights,
    model=model,
    args=args,
    train_dataset=tokenized['train'],
    eval_dataset=tokenized['validation'],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

print("✅ Trainer 설정 완료")
print(f"   - 배치: {args.per_device_train_batch_size}")
print(f"   - 에폭: {args.num_train_epochs}")
print(f"   - 스케줄러: {args.lr_scheduler_type}")

✅ Trainer 설정 완료
   - 배치: 128
   - 에폭: 3
   - 스케줄러: SchedulerType.COSINE


In [13]:
# 이 셀 실행해서 GPU 확인
import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"메모리: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

GPU: Tesla P100-PCIE-16GB
메모리: 17.1 GB


In [14]:
# 학습 실행
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro,F1 Weighted
1,0.871100,0.841424,0.775374,0.609523,0.786741
2,0.639300,0.748178,0.802856,0.669230,0.808531
3,0.529700,0.742387,0.809329,0.681833,0.814534


TrainOutput(global_step=4689, training_loss=0.8266337942431339, metrics={'train_runtime': 2772.8933, 'train_samples_per_second': 216.38, 'train_steps_per_second': 1.691, 'total_flos': 1.6235734105404672e+16, 'train_loss': 0.8266337942431339, 'epoch': 3.0})

In [15]:
# 평가
test_results = trainer.evaluate(tokenized['test'])
print("=== Test Results ===")
for k, v in test_results.items():
    print(f"{k}: {v:.4f}")

=== Test Results ===
eval_loss: 0.7446
eval_accuracy: 0.8110
eval_f1_macro: 0.6774
eval_f1_weighted: 0.8163
eval_runtime: 59.1954
eval_samples_per_second: 595.0470
eval_steps_per_second: 2.3310
epoch: 3.0000


In [16]:
# 모델 저장
save_path = OUTPUT_PATH / f'{TASK}_classifier_final'
trainer.save_model(str(save_path))
tokenizer.save_pretrained(str(save_path))

with open(save_path / 'label_mapping.json', 'w') as f:
    json.dump({TASK: label_mapping[TASK]}, f, ensure_ascii=False, indent=2)

print(f"저장 완료: {save_path}")

저장 완료: /kaggle/working/models/domain_classifier_final


In [17]:
# 추론 테스트
id2label = {int(k): v for k, v in label_mapping[TASK].items()}

def predict(text):
    inputs = tokenizer(text, return_tensors='pt', padding=True, truncation=True, max_length=MAX_LENGTH).to(device)
    model.eval()
    with torch.no_grad():
        probs = torch.softmax(model(**inputs).logits, dim=1)
        pred_id = torch.argmax(probs, dim=1).item()
    return id2label[pred_id], probs[0][pred_id].item()

tests = ["인터넷뱅킹 로그인 오류", "택배 배송 조회", "코로나 검사 장소"]
for t in tests:
    label, conf = predict(t)
    print(f"{t} → {label} ({conf:.1%})")

인터넷뱅킹 로그인 오류 → 금융/보험 (99.0%)
택배 배송 조회 → K쇼핑 (76.8%)
코로나 검사 장소 → 질병관리본부 (96.6%)


In [18]:
# 다운로드용 압축
!zip -r /kaggle/working/{TASK}_classifier.zip {save_path}

  adding: kaggle/working/models/domain_classifier_final/ (stored 0%)
  adding: kaggle/working/models/domain_classifier_final/vocab.txt (deflated 49%)
  adding: kaggle/working/models/domain_classifier_final/tokenizer_config.json (deflated 75%)
  adding: kaggle/working/models/domain_classifier_final/label_mapping.json (deflated 39%)
  adding: kaggle/working/models/domain_classifier_final/training_args.bin (deflated 53%)
  adding: kaggle/working/models/domain_classifier_final/model.safetensors (deflated 7%)
  adding: kaggle/working/models/domain_classifier_final/config.json (deflated 60%)
  adding: kaggle/working/models/domain_classifier_final/tokenizer.json (deflated 69%)
  adding: kaggle/working/models/domain_classifier_final/special_tokens_map.json (deflated 80%)
